In [10]:


from pyfmreader.ps_nex.parseTDMS import grab_tdms
from pyfmreader import loadfile
import matplotlib.pyplot as plt 
from nptdms import TdmsFile #from nptdms import tdms  # pip install nptdms

import pandas as pd
import os 
import shutil

#%%
import seaborn as sns
import shutil
import os

import glob
import os
import time

In [11]:
import os
import shutil
import traceback

def move_to_error_folder(fp, rootdir, error_msg=None, error_type=None, processing_stage="unknown", include_traceback=False):
    """
    Move error files to error folder and return detailed error information.
    
    Args:
        fp: File path
        rootdir: Root directory 
        error_msg: Error message
        error_type: Type of error (e.g., 'FileLoadError', 'ProcessingError')
        processing_stage: Stage where error occurred (e.g., 'file_loading', 'metadata_extraction', 'curve_processing')
        include_traceback: Whether to include full stack trace
    """
    error_folder = os.path.join(rootdir, 'error_file')
    if not os.path.exists(error_folder):
        os.makedirs(error_folder)
    
    # Move files
    try:
        shutil.move(fp, os.path.join(error_folder, os.path.basename(fp)))
        if os.path.exists(fp + '_index'):
            shutil.move(fp + '_index', os.path.join(error_folder, os.path.basename(fp) + '_index'))
    except Exception as move_error:
        print(f"Warning: Could not move file {fp}: {move_error}")
    
    # Capture stack trace if requested
    stack_trace = None
    if include_traceback:
        stack_trace = traceback.format_exc()
    
    if error_msg:
        print(f"ERROR in {processing_stage}: {error_msg}")
    print(f"Moved to error folder: {fp}")

    temp_dict = {
        'file_path': fp,
        'total_len': None,
        'total_len_cal_tick': None,
        'diff_cal': None,
        'diff_store': None,
        'total_len_stored_point': None,
        "store_nbpts_app": None,
        "store_nbpts_con": None,
        "store_nbpts_ret": None,
        "cal_nbpts_app": None,
        "cal_nbpts_con": None,
        "cal_nbpts_ret": None,
        "set_pt_z_pos_V": None,
        "error_message": error_msg,
        "error_type": error_type,
        "processing_stage": processing_stage,
        "stack_trace": stack_trace,
        "has_error": True
    }
    return temp_dict

In [12]:
# Define the directory to search in
directory = '/Users/evillz/Data/article/2025_07_01_THP1_phd/error_files_test'
print(directory)
isMap = False

def check_files_in_directory(directory, isMap=False):
    """ 
    Check all tdms files in directory to see for any errors in the files. If yes, move them to a new folder called 'error_file'.
    
    """

    if isMap:
        # Define the pattern to match files that start with "psnex_map__" and do not end with ".zip"
        pattern = os.path.join(directory, 'psnex_map__*')
        files = [f for f in glob.glob(pattern) if not f.endswith('.zip')]

    else:
        # Get all the folders in the directory including the directory itself
        folders = [f.path for f in os.scandir(directory) if f.is_dir()]
        folders.append(directory)

    print(f'# of folders found : {len(folders)}')

    # start time
    start_time = time.time()

    for rootdir in folders:
        
        # _,all_files,filenames = find_directories_with_file_type(rootdir,'.tdms')
        # Check if there are files inside the folder, if not, move folder into a new folder called 'empty_folders
        print (f"Checking folder: {rootdir}")
        
        try:
            first_file, all_files = grab_tdms(rootdir)
        except FileNotFoundError as e:
            print(f"TDMS FileNotFoundError: {e}")
            continue

        df_point_log = pd.DataFrame()

        tp_file = []
        for fp in all_files[:]:
            print(f'now analyzing {fp}')
            
            # Initialize default values for ALL variables
            temp_dict = {
                'file_path': fp,
                'total_len': None,
                'total_len_cal_tick': None,
                'diff_cal': None,
                'diff_store': None,
                'total_len_stored_point': None,
                "store_nbpts_app": None,
                "store_nbpts_con": None,
                "store_nbpts_ret": None,
                "cal_nbpts_app": None,
                "cal_nbpts_con": None,
                "cal_nbpts_ret": None,
                "set_pt_z_pos_V": None,
            }

            # Try to load the file
            try:
                file = loadfile(fp)
            except (KeyError, TypeError, ValueError, UnboundLocalError) as e:
                move_to_error_folder(fp, rootdir, f"{type(e).__name__}: {e}")
                # Log the error file with None values
                df_temp = pd.DataFrame([temp_dict])  # Note: [temp_dict] not temp_dict
                df_point_log = pd.concat([df_point_log, df_temp], ignore_index=True)
                continue  # Skip to next file
            except Exception as e:
                move_to_error_folder(fp, rootdir, f"Unexpected error loading file: {e}")
                # Log the error file with None values
                df_temp = pd.DataFrame([temp_dict])
                df_point_log = pd.concat([df_point_log, df_temp], ignore_index=True)
                continue  # Skip to next file
            
            # if error in loading file, do not execute this part
            if file is not None:
                try:
                    filemetadata = file.filemetadata
                    #closed_loop = filemetadata['z_closed_loop']
                    file_deflection_sensitivity = filemetadata['defl_sens_nmbyV']  # nm/V
                    file_spring_constant = filemetadata['spring_const_Nbym']  # N/m
                    height_channel = filemetadata['height_channel_key']
                    #force_set_point = filemetadata["force_setpoint"]
                    deflection_sensitivity = filemetadata['defl_sens_nmbyV'] / 1e9  # m/V
                    spring_constant = file_spring_constant
                
                    curve_properties = filemetadata['curve_properties']
                    tick_time_s = filemetadata['instrument_tick_time_(s)']# 2* 10**-6

                    force_curve = file.getcurve(0)
                    # Preprocess curve
                    force_curve.preprocess_force_curve(deflection_sensitivity, height_channel)
                    tdms_file_ps_nex_file = TdmsFile.open(fp)
                    tdms_groups = tdms_file_ps_nex_file.groups() 
                    tdms_psnex_fc = tdms_groups[0]
                    
                    height = tdms_psnex_fc[height_channel][:]
                    total_len = len(height)
                    stored_arr =[]
                    cal_arr = []

                    for i, segment in force_curve.get_segments():
                        temp_seg_dict = curve_properties[str(0)][i]
                        seg_i_pt_cal = temp_seg_dict[f"segment_{i}_nb_points_cal"]
                        seg_i_pt_stored = temp_seg_dict[f"segment_{i}_nb_points_(points)"]
                        stored_arr.append(seg_i_pt_stored)
                        cal_arr.append(seg_i_pt_cal)

                        segment_duration = temp_seg_dict[f"segment_{i}_duration_(ticks)"]*tick_time_s
                        if i ==0:
                            set_pt_z_pos_V = temp_seg_dict[f"segment_{i}_Z_position_setpoint_trigger_(V)"]
                        print(f"segment duration {segment_duration}, \n numb of point cal (ticks, dec, sampling rate {seg_i_pt_cal}, num pts stored per segment {seg_i_pt_stored}")
            
                    # Update temp_dict with successful values
                    temp_dict.update({
                        'total_len': total_len,
                        'total_len_cal_tick': sum(cal_arr),
                        'diff_cal': total_len - sum(cal_arr),
                        'diff_store': total_len - sum(stored_arr),
                        'total_len_stored_point': sum(stored_arr),
                        "store_nbpts_app": stored_arr[0] if stored_arr else None,
                        "store_nbpts_con": stored_arr[1] if len(stored_arr) > 0 else None,
                        "store_nbpts_ret": stored_arr[2] if len(stored_arr) > 1 else None,
                        "cal_nbpts_app": cal_arr[0] if cal_arr else None,
                        "cal_nbpts_con": cal_arr[1] if len(cal_arr) > 0 else None,
                        "cal_nbpts_ret": cal_arr[2] if len(cal_arr) > 1 else None,
                        "set_pt_z_pos_V": set_pt_z_pos_V if 'set_pt_z_pos_V' in locals() else None
                    })
                    
                except Exception as e:
                    move_to_error_folder(fp, rootdir, f"Error processing file: {e}")
                    # temp_dict already has None values, so we're good
                    
            # ALWAYS log the file (moved inside the loop!)
            df_temp = pd.DataFrame([temp_dict])
            df_point_log = pd.concat([df_point_log, df_temp], ignore_index=True)

        # Save results for this folder (this stays outside the file loop)
        if not df_point_log.empty:
            results_folder = os.path.join(rootdir, 'results')
            if not os.path.exists(results_folder):
                os.makedirs(results_folder)
            df_point_log.to_csv(os.path.join(results_folder, 'df_point_log.csv'), index=False)

    # Print the total time taken
    end_time = time.time()
    print(f"Total time taken: {end_time - start_time} seconds")


check_files_in_directory(directory, isMap)
# %%

/Users/evillz/Data/article/2025_07_01_THP1_phd/error_files_test
# of folders found : 16
Checking folder: /Users/evillz/Data/article/2025_07_01_THP1_phd/error_files_test/300
now analyzing /Users/evillz/Data/article/2025_07_01_THP1_phd/error_files_test/300/fcurve_thp1_cell1_ret_300ums__2025.07.01_16.48.30.15.tdms
decimation: [10 10 10]
z_sensor_delay: 0.001, bool_correct_overshoot: True 
points removed : 50
start_indices: [    0 48232 53560]
end_indices: [48232 53560 62546]
NumPnts: [48232  5328  8986]
Deflection Length: 62496, Height: 62496
0 48232
segment relative position: 48232, segment duration: 0.96464175
App
overshoot in the approach, accounted for   
removed overshoot
---------------------
53501 58828
segment relative position: 5327, segment duration: 0.106564525
Con
---------------------
53560 62496
segment relative position: 8936, segment duration: 0.17972145
Ret
---------------------
segment duration 0.96464175, 
 numb of point cal (ticks, dec, sampling rate 48232, num pts sto

/var/folders/kc/jykhdgwx4p3696y51hzh6j3r0000gn/T/ipykernel_32457/401380172.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_point_log = pd.concat([df_point_log, df_temp], ignore_index=True)
/var/folders/kc/jykhdgwx4p3696y51hzh6j3r0000gn/T/ipykernel_32457/401380172.py:75: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_point_log = pd.concat([df_point_log, df_temp], ignore_index=True)


decimation: [  10   10 1000]
z_sensor_delay: 0.001, bool_correct_overshoot: True 
points removed : 50
start_indices: [    0 43778 44843]
end_indices: [43778 44843 47699]
NumPnts: [43778  1065  2856]
Deflection Length: 47650, Height: 47650
0 43778
segment relative position: 43778, segment duration: 0.87556675
App
overshoot in the approach, accounted for   
removed overshoot
---------------------
44804 45868
segment relative position: 1064, segment duration: 0.021314275
Con
---------------------
44843 47650
segment relative position: 2807, segment duration: 5.7129625
Ret
---------------------
segment duration 0.87556675, 
 numb of point cal (ticks, dec, sampling rate 43778, num pts stored per segment 437783
segment duration 0.021314275, 
 numb of point cal (ticks, dec, sampling rate 1065, num pts stored per segment 10656
segment duration 5.7129625, 
 numb of point cal (ticks, dec, sampling rate 2856, num pts stored per segment 2856479
now analyzing /Users/evillz/Data/article/2025_07_01_T

/var/folders/kc/jykhdgwx4p3696y51hzh6j3r0000gn/T/ipykernel_32457/401380172.py:69: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_point_log = pd.concat([df_point_log, df_temp], ignore_index=True)


removed overshoot
---------------------
4535 4535
segment relative position: 0, segment duration: 1.4499999999999999e-06
Con
---------------------
4508 13493
segment relative position: 8985, segment duration: 1.7980005
Ret
---------------------
segment duration 0.90163725, 
 numb of point cal (ticks, dec, sampling rate 4508, num pts stored per segment 450818
segment duration 1.4499999999999999e-06, 
 numb of point cal (ticks, dec, sampling rate 0, num pts stored per segment 0
segment duration 1.7980005, 
 numb of point cal (ticks, dec, sampling rate 8990, num pts stored per segment 898999
now analyzing /Users/evillz/Data/article/2025_07_01_THP1_phd/error_files_test/10/fcurve_thp1_cell1_ret_100ums__2025.07.01_16.33.22.40.tdms
decimation: [100 100 100]
z_sensor_delay: 0.001, bool_correct_overshoot: True 
points removed : 5
start_indices: [   0 4526 4526]
end_indices: [ 4526  4526 13508]
NumPnts: [4526    1 8982]
Deflection Length: 13503, Height: 13503
0 4526
segment relative position: 45

/var/folders/kc/jykhdgwx4p3696y51hzh6j3r0000gn/T/ipykernel_32457/401380172.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_point_log = pd.concat([df_point_log, df_temp], ignore_index=True)
/var/folders/kc/jykhdgwx4p3696y51hzh6j3r0000gn/T/ipykernel_32457/401380172.py:75: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_point_log = pd.concat([df_point_log, df_temp], ignore_index=True)


decimation: [100 100 100]
z_sensor_delay: 0.001, bool_correct_overshoot: True 
points removed : 5
start_indices: [   0 4261 4793]
end_indices: [ 4261  4793 13324]
NumPnts: [4261  532 8531]
Deflection Length: 13321, Height: 13321
0 4261
segment relative position: 4261, segment duration: 0.8523864999999999
App
overshoot in the approach, accounted for   
removed overshoot
---------------------
4803 5334
segment relative position: 531, segment duration: 0.106564175
Con
---------------------
4793 13321
segment relative position: 8528, segment duration: 1.7063997499999999
Ret
---------------------
segment duration 0.8523864999999999, 
 numb of point cal (ticks, dec, sampling rate 4261, num pts stored per segment 426193
segment duration 0.106564175, 
 numb of point cal (ticks, dec, sampling rate 532, num pts stored per segment 53281
segment duration 1.7063997499999999, 
 numb of point cal (ticks, dec, sampling rate 8531, num pts stored per segment 853198
now analyzing /Users/evillz/Data/artic

/var/folders/kc/jykhdgwx4p3696y51hzh6j3r0000gn/T/ipykernel_32457/401380172.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_point_log = pd.concat([df_point_log, df_temp], ignore_index=True)


now analyzing /Users/evillz/Data/article/2025_07_01_THP1_phd/error_files_test/1000_3/fcurve_thp1_cell1_ret_1000ums__2025.07.01_17.02.55.32.tdms
decimation: [1 1 1]
z_sensor_delay: 0.001, bool_correct_overshoot: True 
points removed : 500
start_indices: [     0 473878 484535]
end_indices: [473878 484535 578737]
NumPnts: [473878  10657  94202]
Deflection Length: 493517, Height: 493517
0 473878
segment relative position: 473878, segment duration: 0.9477565
App
overshoot in the approach, accounted for   
removed overshoot
---------------------
399178 409834
segment relative position: 10656, segment duration: 0.0213146
Con
---------------------
484535 493517
segment relative position: 8982, segment duration: 0.18840537499999999
Ret
---------------------
segment duration 0.9477565, 
 numb of point cal (ticks, dec, sampling rate 473878, num pts stored per segment 473878
segment duration 0.0213146, 
 numb of point cal (ticks, dec, sampling rate 10657, num pts stored per segment 10656
segment d

/var/folders/kc/jykhdgwx4p3696y51hzh6j3r0000gn/T/ipykernel_32457/401380172.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_point_log = pd.concat([df_point_log, df_temp], ignore_index=True)
/var/folders/kc/jykhdgwx4p3696y51hzh6j3r0000gn/T/ipykernel_32457/401380172.py:69: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_point_log = pd.concat([df_point_log, df_temp], ignore_index=True)
